In [0]:
import pandas as pd

silver_base = "/Volumes/customersprocess/default/customer_sales_silver"

customer_path = f"{silver_base}/customer.csv"
orders_path = f"{silver_base}/orders.csv"
sales_path = f"{silver_base}/sales.csv"

customer_df = pd.read_csv(customer_path)
orders_df = pd.read_csv(orders_path)
sales_df = pd.read_csv(sales_path)

print("Silver files loaded successfully")

In [0]:
print("Customer:", customer_df.shape)
print("Orders:", orders_df.shape)
print("Sales:", sales_df.shape)

In [0]:
orders_df["order_date"] = pd.to_datetime(
    orders_df["order_date"],
    errors="coerce"
)

print(orders_df["order_date"].dtype)

Here I am joining the customer and orders DataFrames using customer_id. One customer can have multiple orders, so this is a one-to-many relationship. I use an inner join because I only want records where a valid customer and order relationship exists. The validate="one_to_many" parameter also allows Pandas to check whether the relationship follows the structure that I expect.

In [0]:
customer_orders_df = customer_df.merge(
    orders_df,
    on="customer_id",
    how="inner",
    validate="one_to_many"
)

print("Customer + Orders joined")
print("Rows:", len(customer_orders_df))
print("Columns:", customer_orders_df.columns.tolist())

After joining customers and orders, I join the result with the sales DataFrame using order_id. In our use case, each order should have one sales record, so I use one-to-one validation. This helps detect unexpected duplicate sales records instead of allowing incorrect data to silently enter the Gold dataset.

In [0]:
gold_df = customer_orders_df.merge(
    sales_df,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

print("Customer + Orders + Sales joined")
print("Gold rows:", len(gold_df))
print("Gold columns:", gold_df.columns.tolist())

In [0]:
gold_df.head()

Here I am creating a new derived column called gross_amount. I calculate it by multiplying unit_price by quantity. This is a derived business field because it was not directly available in the source data. It represents the gross value of the ordered quantity before considering the discount.

In [0]:
gold_df["gross_amount"] = (
    gold_df["unit_price"] * gold_df["quantity"]
)

print("Gross amount calculated")

In [0]:
gold_df[
    [
        "order_id",
        "product_id_x",
        "quantity",
        "unit_price",
        "discount",
        "sales_amount",
        "gross_amount"
    ]
].head(10)

In [0]:
if gold_df.empty:

    print("FAIL: Gold dataset is empty")
    gold_dq_passed = False

else:

    print(
        "PASS: Gold dataset contains",
        len(gold_df),
        "rows"
    )

    gold_dq_passed = True

In [0]:
required_gold_columns = [
    "customer_id",
    "customer_name",
    "order_id",
    "product_id_x",
    "quantity",
    "unit_price",
    "sales_amount"
]

for column in required_gold_columns:

    null_count = gold_df[column].isnull().sum()

    if null_count > 0:

        print(
            f"FAIL: {column} contains "
            f"{null_count} null values"
        )

        gold_dq_passed = False

    else:

        print(
            f"PASS: {column} contains no null values"
        )

In [0]:
duplicate_orders = gold_df["order_id"].duplicated().sum()

if duplicate_orders > 0:

    print(
        "FAIL: Gold contains",
        duplicate_orders,
        "duplicate order IDs"
    )

    gold_dq_passed = False

else:

    print(
        "PASS: Gold order_id is unique"
    )

In [0]:
gold_csv_path = (
    "/Volumes/customersprocess/default/"
    "customer_sales_gold/gold_data.csv"
)

gold_df.to_csv(
    gold_csv_path,
    index=False
)

print("Gold CSV written successfully")

After completing the joins and creating the derived columns, I write the final Gold DataFrame to a CSV file in the Gold volume. This file is then used to create the permanent Databricks Gold table.

In [0]:
%sql
CREATE OR REPLACE TABLE customersprocess.default.customer_sales_gold
AS
SELECT *
FROM read_files(
    '/Volumes/customersprocess/default/customer_sales_gold/gold_data.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => true
);

In [0]:
%sql
CREATE OR REPLACE TABLE customersprocess.default.customer_sales_gold
USING DELTA
AS
SELECT *
FROM csv.`/Volumes/customersprocess/default/customer_sales_gold/gold_data.csv`

In [0]:
from datetime import datetime
import pandas as pd

def log_pipeline_result(stage, status, message):

    timestamp = datetime.now()
    timestamp_text = timestamp.strftime("%Y%m%d_%H%M%S")

    log_path = (
        "/Volumes/customersprocess/default/"
        f"customer_sales_logs/"
        f"{stage}_{status}_{timestamp_text}.csv"
    )

    log_data = pd.DataFrame([
        {
            "timestamp": timestamp,
            "pipeline": "customer_sales_etl",
            "stage": stage,
            "status": status,
            "message": message
        }
    ])

    log_data.to_csv(log_path, index=False)

    print(f"Log written: {log_path}")

In [0]:
try:

    # Your existing Gold code
    # Read Silver
    # Convert order_date
    # Join customer + orders
    # Join sales
    # Calculate gross_amount
    # Run Gold DQ
    # Write gold_data.csv

    log_pipeline_result(
        "gold",
        "SUCCESS",
        "Gold transformation completed successfully"
    )

except Exception as e:

    log_pipeline_result(
        "gold",
        "FAILED",
        str(e)
    )

    raise